# Spaceship Titanic | Notebook 04: Spending Feature Engineering

**Project stage:** Controlled feature-engineering experiment (CAT-002)  
**Goal:** Test whether spending summaries improve CatBoost classification beyond the 13-feature baseline.

This notebook is self-contained: it reloads the competition CSVs, rebuilds engineered variables, and evaluates them on the development-training subset. It **does not replace** the submitted CatBoost baseline.

In [1]:
# ============================================================
# STEP 17A: IMPORT LIBRARIES AND LOAD DATA
# ============================================================

# Import pandas for data manipulation.
import pandas as pd

# Import NumPy for numerical operations.
import numpy as np

# Import Path for managing file locations.
from pathlib import Path


# Locate the competition files without relying on a personal Windows path.
# Supports launching the notebook in the project root or in notebooks/.
current_dir = Path.cwd()
possible_roots = [current_dir, current_dir.parent]
project_root = next(
    (folder for folder in possible_roots
     if (folder / "train.csv").is_file()
     and (folder / "test.csv").is_file()),
    None
)
if project_root is None:
    raise FileNotFoundError(
        "Could not locate train.csv and test.csv in the current "
        "directory or its parent. Place both Kaggle CSV files in "
        "the Spaceship Titanic project root."
    )
print("Using competition data directory:", project_root)

# Load the original training dataset.
train_df = pd.read_csv(
    project_root / "train.csv"
)

# Load Kaggle's test dataset.
test_df = pd.read_csv(
    project_root / "test.csv"
)

# Confirm dataset dimensions.
print("Training dataset:", train_df.shape)
print("Test dataset:", test_df.shape)

Using competition data directory: C:\Users\A\Desktop\sample_project_1\spaceship-titanic
Training dataset: (8693, 14)
Test dataset: (4277, 13)


## 1. Review the Five Spending Variables

We first inspect onboard spending (`RoomService`, `FoodCourt`, `ShoppingMall`, `Spa`, `VRDeck`) and their missing values. Missing expenditure is not automatically equivalent to zero expenditure.

In [2]:
# ============================================================
# STEP 17B: IDENTIFY SPENDING VARIABLES
# ============================================================

# Define the five onboard expenditure columns.
spending_columns = [
    "RoomService",
    "FoodCourt",
    "ShoppingMall",
    "Spa",
    "VRDeck"
]

# Display the first ten passengers' spending values.
print("First ten training passengers:")

display(
    train_df[spending_columns].head(10)
)

# Count missing values in each spending column.
print("\nMissing values in spending columns:")

print(
    train_df[spending_columns].isna().sum()
)

First ten training passengers:


,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck
0,0.0,0.0,0.0,0.0,0.0
1,109.0,9.0,25.0,549.0,44.0
2,43.0,3576.0,0.0,6715.0,49.0
3,0.0,1283.0,371.0,3329.0,193.0
4,303.0,70.0,151.0,565.0,2.0
5,0.0,483.0,0.0,291.0,0.0
6,42.0,1539.0,3.0,0.0,0.0
7,0.0,0.0,0.0,0.0,NaN
8,0.0,785.0,17.0,216.0,0.0
9,0.0,0.0,0.0,0.0,0.0



Missing values in spending columns:
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
dtype: int64


## 2. Define Missingness-Aware Spending Features

We introduce three derived predictors: **TotalSpending** (defined only when all five amounts are known), **SpendingMissingCount**, and **SpendingStatus**. The status is *Positive Spending* if any observed charge is positive, *Zero Spending* only if all five charges are observed and zero, and *Unknown* otherwise. A positive charge establishes positive spending even if some other charges are missing.

In [3]:
# ============================================================
# STEP 17C: CREATE SPENDING FEATURES
# ============================================================

def engineer_spending_features(df):
    """
    Create three features describing passenger spending:

    1. TotalSpending:
       Total across five onboard services.
       Missing if any spending component is missing.

    2. SpendingStatus:
       Positive Spending, Zero Spending, or Unknown.

    3. SpendingMissingCount:
       Number of missing spending observations.
    """

    # Create a copy to avoid modifying the original data.
    df = df.copy()

    # Define the spending columns.
    spending_columns = [
        "RoomService",
        "FoodCourt",
        "ShoppingMall",
        "Spa",
        "VRDeck"
    ]

    # --------------------------------------------------------
    # FEATURE 1: COUNT MISSING SPENDING VALUES
    # --------------------------------------------------------

    # Count how many spending columns are missing
    # for each passenger.
    df["SpendingMissingCount"] = (
        df[spending_columns].isna().sum(axis=1)
    )

    # --------------------------------------------------------
    # FEATURE 2: TOTAL SPENDING
    # --------------------------------------------------------

    # Sum spending values only when all five values
    # are available.
    #
    # min_count=5 ensures that a passenger with any
    # missing spending component receives NaN.
    df["TotalSpending"] = (
        df[spending_columns].sum(
            axis=1,
            min_count=5
        )
    )

    # --------------------------------------------------------
    # FEATURE 3: SPENDING STATUS
    # --------------------------------------------------------

    # Check whether at least one observed spending
    # value is greater than zero.
    has_positive_spending = (
        df[spending_columns].gt(0).any(axis=1)
    )

    # Check whether all five spending values are known.
    all_spending_known = (
        df[spending_columns].notna().all(axis=1)
    )

    # Check whether all five values are exactly zero.
    has_zero_spending = (
        all_spending_known
        & df[spending_columns].eq(0).all(axis=1)
    )

    # Assign categories based on the evidence available.
    #
    # Positive Spending:
    # At least one spending value is greater than zero.
    #
    # Zero Spending:
    # All five spending values are observed and equal zero.
    #
    # Unknown:
    # Spending information is incomplete and no
    # positive amount has been observed.
    df["SpendingStatus"] = np.select(
        [
            has_positive_spending,
            has_zero_spending
        ],
        [
            "Positive Spending",
            "Zero Spending"
        ],
        default="Unknown"
    )

    # Return the transformed dataset.
    return df


print("Spending feature engineering function created!")

Spending feature engineering function created!


## 3. Apply Transformations to Both Datasets

The same deterministic feature engineering function is applied to labelled training records and unlabeled competition test records. No target labels are used in the transformation.

In [4]:
# ============================================================
# STEP 17D: APPLY SPENDING FEATURE ENGINEERING
# ============================================================

# Create spending features for training passengers.
train_fe = engineer_spending_features(train_df)

# Create spending features for Kaggle test passengers.
test_fe = engineer_spending_features(test_df)

# Display the resulting dataset dimensions.
print(
    "Training dataset after feature engineering:",
    train_fe.shape
)

print(
    "Test dataset after feature engineering:",
    test_fe.shape
)

# Display the new features for the first ten passengers.
display(
    train_fe[
        spending_columns
        + [
            "TotalSpending",
            "SpendingStatus",
            "SpendingMissingCount"
        ]
    ].head(10)
)

Training dataset after feature engineering: (8693, 17)
Test dataset after feature engineering: (4277, 16)


,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,TotalSpending,SpendingStatus,SpendingMissingCount
0,0.0,0.0,0.0,0.0,0.0,0.0,Zero Spending,0
1,109.0,9.0,25.0,549.0,44.0,736.0,Positive Spending,0
2,43.0,3576.0,0.0,6715.0,49.0,10383.0,Positive Spending,0
3,0.0,1283.0,371.0,3329.0,193.0,5176.0,Positive Spending,0
4,303.0,70.0,151.0,565.0,2.0,1091.0,Positive Spending,0
5,0.0,483.0,0.0,291.0,0.0,774.0,Positive Spending,0
6,42.0,1539.0,3.0,0.0,0.0,1584.0,Positive Spending,0
7,0.0,0.0,0.0,0.0,NaN,NaN,Unknown,1
8,0.0,785.0,17.0,216.0,0.0,1018.0,Positive Spending,0
9,0.0,0.0,0.0,0.0,0.0,0.0,Zero Spending,0


## 4. Validate Engineered Feature Logic

These assertions check that partially missing spending never yields a complete `TotalSpending`, that complete records have defined totals, and that the `Zero Spending` category is consistent with the numerical total. We also review the status distribution.

In [5]:
# ============================================================
# STEP 17E: VALIDATE ENGINEERED FEATURES
# ============================================================

# Display the frequency distribution of SpendingStatus.
print("SPENDING STATUS DISTRIBUTION")
print("-" * 50)

print(
    train_fe["SpendingStatus"].value_counts()
)

# Display the distribution of missing spending counts.
print("\nNUMBER OF MISSING SPENDING VALUES")
print("-" * 50)

print(
    train_fe["SpendingMissingCount"]
    .value_counts()
    .sort_index()
)

# Confirm that passengers with any missing spending
# component have an unknown complete spending total.
missing_component = (
    train_fe["SpendingMissingCount"] > 0
)

assert train_fe.loc[
    missing_component,
    "TotalSpending"
].isna().all(), (
    "Some incomplete spending records have numeric totals."
)

# Confirm that complete spending records have a total.
complete_component = (
    train_fe["SpendingMissingCount"] == 0
)

assert train_fe.loc[
    complete_component,
    "TotalSpending"
].notna().all(), (
    "Some complete spending records have missing totals."
)

# Confirm that all zero-spending passengers have
# exactly zero total spending.
zero_spending = (
    train_fe["SpendingStatus"] == "Zero Spending"
)

assert (
    train_fe.loc[zero_spending, "TotalSpending"] == 0
).all(), (
    "Zero-spending classification is inconsistent."
)

print("\nAll feature engineering validation checks passed!")

SPENDING STATUS DISTRIBUTION
--------------------------------------------------
SpendingStatus
Positive Spending    5040
Zero Spending        3247
Unknown               406
Name: count, dtype: int64

NUMBER OF MISSING SPENDING VALUES
--------------------------------------------------
SpendingMissingCount
0    7785
1     873
2      35
Name: count, dtype: int64

All feature engineering validation checks passed!


## 5. Extract Structured Cabin Information

As in the baseline experiment, split `Cabin` into deck, numeric cabin number, and side. This ensures the spending-feature model is compared with the same underlying cabin information as CAT-001.

In [6]:
# ============================================================
# STEP 18A: EXTRACT CABIN FEATURES
# ============================================================

def extract_cabin_features(df):
    """
    Extract three features from Cabin:

    1. CabinDeck
    2. CabinNumber
    3. CabinSide
    """

    # Create a copy to preserve the original dataset.
    df = df.copy()

    # Split Cabin into its three components.
    cabin_parts = df["Cabin"].str.split("/", expand=True)

    # Extract the deck.
    df["CabinDeck"] = cabin_parts[0]

    # Extract the cabin number as a numeric feature.
    df["CabinNumber"] = pd.to_numeric(
        cabin_parts[1],
        errors="coerce"
    )

    # Extract the side.
    df["CabinSide"] = cabin_parts[2]

    # Remove the original Cabin column.
    df = df.drop(columns=["Cabin"])

    return df


# Apply the transformation to both datasets.
train_fe = extract_cabin_features(train_fe)

test_fe = extract_cabin_features(test_fe)

print("Cabin features extracted successfully!")

print("Training dataset:", train_fe.shape)
print("Test dataset:", test_fe.shape)

Cabin features extracted successfully!
Training dataset: (8693, 19)
Test dataset: (4277, 18)


## 6. Construct the CAT-002 Predictor Set

CAT-002 uses the original **13 predictors** plus three engineered spending features (**16 predictors** in total). `Transported` remains the binary target. The CSV test set is prepared for possible predictions but is not used to fit the cross-validation models.

In [7]:
# ============================================================
# STEP 18B: DEFINE OUR FEATURE SET
# ============================================================

# Original numerical features.
numerical_features = [
    "Age",
    "RoomService",
    "FoodCourt",
    "ShoppingMall",
    "Spa",
    "VRDeck",
    "CabinNumber"
]

# Original categorical features.
categorical_features = [
    "HomePlanet",
    "CryoSleep",
    "Destination",
    "VIP",
    "CabinDeck",
    "CabinSide"
]

# New numerical spending features.
new_numerical_features = [
    "TotalSpending",
    "SpendingMissingCount"
]

# New categorical spending feature.
new_categorical_features = [
    "SpendingStatus"
]

# Combine the original and engineered predictors.
selected_features = (
    numerical_features
    + categorical_features
    + new_numerical_features
    + new_categorical_features
)

# Update the categorical feature list for CatBoost.
cat_features = (
    categorical_features
    + new_categorical_features
)

# Select predictors and target.
X = train_fe[selected_features].copy()

y = train_fe["Transported"].copy()

# Prepare features for future Kaggle predictions.
X_test = test_fe[selected_features].copy()

# Display dimensions.
print("Training features:", X.shape)
print("Target:", y.shape)
print("Test features:", X_test.shape)

print("Total predictors:", len(selected_features))

Training features: (8693, 16)
Target: (8693,)
Test features: (4277, 16)
Total predictors: 16


## 7. Reproduce the Development Split

Use the same stratified 80/20 split (`random_state=42`) used in prior experiments. **Cross-validation is performed on the 6,954 development-training observations**, not the 1,739-observation validation subset.

In [8]:
# ============================================================
# STEP 18C: REPRODUCE OUR ORIGINAL DATA SPLIT
# ============================================================

# Import the splitting function.
from sklearn.model_selection import train_test_split

# Split the data using the same parameters
# as our original experiment.
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Confirm dataset dimensions.
print("Training observations:", X_train.shape)
print("Validation observations:", X_valid.shape)

Training observations: (6954, 16)
Validation observations: (1739, 16)


## 8. Prepare Categorical Predictors for CatBoost

CatBoost accepts categorical feature names and handles numerical `NaN` natively. Here, categorical missing values are represented by the literal string `Missing`. Numerical missing values, including incomplete `TotalSpending`, are kept as `NaN`.

In [9]:
# ============================================================
# STEP 18D: PREPARE CATBOOST CATEGORICAL VARIABLES
# ============================================================

# Create a copy of the training predictors.
X_train_cat = X_train.copy()

# Create a copy of the validation predictors.
X_valid_cat = X_valid.copy()

# Convert categorical values into strings.
# Missing categorical values become "Missing".
for column in cat_features:

    X_train_cat[column] = (
        X_train_cat[column]
        .astype("string")
        .fillna("Missing")
        .astype(str)
    )

    X_valid_cat[column] = (
        X_valid_cat[column]
        .astype("string")
        .fillna("Missing")
        .astype(str)
    )

# Check missing categorical values.
print(
    "Missing categorical values in training:",
    X_train_cat[cat_features].isna().sum().sum()
)

print(
    "Missing categorical values in validation:",
    X_valid_cat[cat_features].isna().sum().sum()
)

Missing categorical values in training: 0
Missing categorical values in validation: 0


## 9. Evaluate CAT-002 with Five-Fold Cross-Validation

Use five stratified folds with the baseline hyperparameters held fixed (`iterations=500`, `depth=6`, `learning_rate=0.05`). This comparison isolates the tested feature-set change as far as the controlled development setup allows. Fold-level standard deviation describes variation across folds, not a confidence interval.

In [10]:
# ============================================================
# STEP 18E: FIVE-FOLD CATBOOST CROSS-VALIDATION
# ============================================================

# Import required libraries.
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from catboost import CatBoostClassifier

# Define the same five-fold strategy used previously.
cv_strategy = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Store validation accuracy for each fold.
cat002_cv_scores = []


# Evaluate CatBoost across five folds.
for fold_number, (train_idx, valid_idx) in enumerate(
    cv_strategy.split(X_train_cat, y_train),
    start=1
):

    # Select the training observations for this fold.
    X_fold_train = X_train_cat.iloc[train_idx].copy()

    # Select the validation observations for this fold.
    X_fold_valid = X_train_cat.iloc[valid_idx].copy()

    # Select corresponding target values.
    y_fold_train = y_train.iloc[train_idx]
    y_fold_valid = y_train.iloc[valid_idx]

    # Create a new CatBoost model for this fold.
    # Hyperparameters remain identical to CAT-001.
    fold_model = CatBoostClassifier(
        iterations=500,
        depth=6,
        learning_rate=0.05,
        loss_function="Logloss",
        random_seed=42,
        verbose=False
    )

    # Train the model.
    fold_model.fit(
        X_fold_train,
        y_fold_train,
        cat_features=cat_features
    )

    # Predict the validation observations.
    fold_predictions = (
        fold_model.predict(X_fold_valid).ravel()
    )

    # Calculate validation accuracy.
    fold_accuracy = accuracy_score(
        y_fold_valid,
        fold_predictions
    )

    # Save the accuracy.
    cat002_cv_scores.append(fold_accuracy)

    # Display the current fold's accuracy.
    print(
        f"Fold {fold_number}: "
        f"{fold_accuracy * 100:.2f}%"
    )


# Convert the scores into a NumPy array.
cat002_cv_scores = np.array(cat002_cv_scores)

# Display summary statistics.
print("\nCAT-002 CROSS-VALIDATION SUMMARY")
print("-" * 50)

print(
    f"Mean CV Accuracy: "
    f"{cat002_cv_scores.mean() * 100:.2f}%"
)

print(
    f"CV Standard Deviation: "
    f"{cat002_cv_scores.std() * 100:.2f} percentage points"
)

Fold 1: 81.09%
Fold 2: 81.45%
Fold 3: 80.66%
Fold 4: 80.45%
Fold 5: 81.37%

CAT-002 CROSS-VALIDATION SUMMARY
--------------------------------------------------
Mean CV Accuracy: 81.00%
CV Standard Deviation: 0.39 percentage points


## 10. Compare Against the 13-Feature Baseline

Compare CAT-002 to previously recorded CAT-001 fold scores. **Note:** CAT-001 scores below were copied in rounded form from the earlier notebook, so its reported standard deviation (0.29 pp) can differ slightly from values calculated using unrounded folds (about 0.28 pp).

In [11]:
# ============================================================
# STEP 18F: COMPARE BASELINE AND ENGINEERED FEATURES
# ============================================================

# CAT-001 fold accuracies from our previous experiment.
cat001_cv_scores = np.array([
    0.8081,
    0.8109,
    0.8124,
    0.8160,
    0.8086
])

# Create a comparison table.
feature_comparison = pd.DataFrame({
    "Experiment": [
        "CAT-001 Baseline",
        "CAT-002 Spending Features"
    ],

    "Number_of_Features": [
        13,
        16
    ],

    "Mean_CV_Accuracy": [
        cat001_cv_scores.mean(),
        cat002_cv_scores.mean()
    ],

    "CV_Std": [
        cat001_cv_scores.std(),
        cat002_cv_scores.std()
    ]
})

# Convert accuracy proportions to percentages.
feature_comparison["Mean_CV_Accuracy"] *= 100
feature_comparison["CV_Std"] *= 100

# Display results.
display(feature_comparison.round(2))

# Calculate the difference in average accuracy.
accuracy_improvement = (
    cat002_cv_scores.mean()
    - cat001_cv_scores.mean()
) * 100

print(
    f"\nChange in CV Accuracy: "
    f"{accuracy_improvement:+.2f} percentage points"
)

,Experiment,Number_of_Features,Mean_CV_Accuracy,CV_Std
0,CAT-001 Baseline,13,81.12,0.29
1,CAT-002 Spending Features,16,81.00,0.39



Change in CV Accuracy: -0.12 percentage points


## 11. Interpretation and Decision

**Observed five-fold development CV:** CAT-001 (13 features) reached **81.12%** mean accuracy; CAT-002 (16 features) reached **81.00%** (**−0.12 percentage points**). This experiment did **not** demonstrate improvement from the three added spending features. The result does not imply that spending information is irrelevant: the baseline already includes the original five spending variables.

**Decision:** Retain the original 13-feature CatBoost configuration as the currently preferred baseline. Avoid reporting the Kaggle test-set score as a result of CAT-002, because no CAT-002 leaderboard submission was performed in this notebook.

**Limitations:** Repeated feature experiments on the same development folds can lead to optimistic model selection; fold-to-fold standard deviations do not quantify all generalization uncertainty. No causal claims can be inferred from these predictive results.

**Reproducibility:** Run from a fresh kernel with `train.csv` and `test.csv` in the notebook's current directory or its parent. CAT-001 comparison scores are transcribed from the preceding model-development notebook.